# MBMF Q1 — Version 8 publication runner

**Scientific invariant:** `mbmf_q1_final_v7.py` is frozen byte-for-byte. V8 is infrastructure-only.

In [ ]:
from pathlib import Path
import copy
import pandas as pd
import mbmf_q1_final_v7 as core
import mbmf_q1_infrastructure_v8 as pub

RUN_SELF_TEST = True
RUN_FULL = False
RUN_ABLATION = False
RUN_HYPERPARAMETER_SCAN = False
RUN_ALL_DATASETS = False
RUN_EXTERNAL_MERGE = False

print("Frozen V7 core SHA256:", pub.FROZEN_V7_CORE_SHA256)
if RUN_SELF_TEST:
    pub.publication_extension_self_test_v8()

## Primary configuration

In [ ]:
cfg = pub.PublicationExperimentConfigV8()
cfg.data.data_root = "./datasets"
cfg.data.dataset = "ml-1m"
cfg.output_dir = "./results_q1_v8/ml-1m"

# Main paper grid
cfg.core_config.split_seeds = (2026, 2027, 2028, 2029, 2030)
cfg.core_config.model_seeds = (11, 22, 33, 44, 55)
cfg.core_config.latent_dims = (10, 20, 50)

# MBMF-F/C/N initialization study
cfg.core_config.paper.init_modes = (
    "legacy_uniform_2pi",
    "canonical_uniform",
    "isotropic_gaussian",
)

# Proposed Methods 1-7 are exactly the frozen V7 definitions.
cfg.methods = (
    "BiasMF", "MF", "MBMF-F", "MBMF-C", "MBMF-N",
    "AutoRec", "AEMC-NE",
    "Method1", "Method2", "Method3", "Method4", "Method5", "Method6", "Method7",
)

cfg.metrics.ranking_ks = (1, 5, 10, 20, 50)
cfg

## Full common-protocol run
Candidate models see train+validation metrics only. Test metrics are unlocked only after validation selection.

In [ ]:
if RUN_FULL:
    candidates, locked = pub.run_publication_study_v8(cfg)
    display(locked.head())

## All six datasets

In [ ]:
if RUN_ALL_DATASETS:
    all_runs, all_summary = pub.run_all_datasets_v8(
        cfg,
        datasets=("synthetic","jester","bookcrossing","ml-100k","ml-1m","ml-10m"),
        output_root="./results_q1_v8/all_datasets",
    )
    display(all_summary.head())

## Ablation suite
V8 removes the two duplicate V7 ablations and uses 38 unique mechanism-level jobs.

In [ ]:
if RUN_ABLATION:
    ablation_runs = pub.run_ablation_suite_v8(cfg, "./results_q1_v8/ablations")
    display(ablation_runs[["ablation","method","test_RMSE"]].head())

## Hyperparameter one-factor scans
Separate from mechanism ablations; all settings are validation-selected.

In [ ]:
if RUN_HYPERPARAMETER_SCAN:
    plan = pub.HyperparameterScanPlanV8()
    jobs = pub.make_hyperparameter_scan_jobs_v8(cfg, plan)
    print("scan jobs:", len(jobs))
    # hyper_runs = pub.run_hyperparameter_scan_v8(cfg, plan, "./results_q1_v8/hyperparameter_scans")

## External author implementations
For BMC-ADMM, Deep-MMMF and DRPRA use the exact exported split, then ingest `split,userId,movieId,prediction`.

In [ ]:
# Example preparation for an external implementation:
# raw, spec = pub.load_publication_dataset(cfg.data, save_mappings_to="./results_q1_v8/id_maps")
# dc = copy.deepcopy(cfg.data); dc.split_seed = 2026
# split = pub.make_core_split(raw, spec, dc)
# data, backend = pub.build_publication_matrix_data_v8(split, spec, dc, core.get_device())
# pub.export_split_for_external(data, "./external/common_split.csv")
#
# After external predictions exist:
# row = pub.evaluate_external_predictions_v8(
#     data, "./external/bmc_admm_predictions.csv", "BMC-ADMM-official", cfg.metrics
# )

## Loading a selected checkpoint

In [ ]:
# selected_path = "./results_q1_v8/ml-1m/selected_models/...pt"
# model, metadata = pub.load_model_checkpoint_v8(selected_path, data, core.get_device())
# print(metadata["family"], metadata["data_signature"])

## Output checklist
The study writes candidate validation runs, selected train/val/test metrics, persistent selected checkpoints, prediction archives, stability tables, statistics, PNG+PDF figures, split files, ID maps, config/environment manifests and training histories.